# Follow-up: normalising the images, and refitting with changed images (nearest neighbour)

Three objects (transistor, pill, metal_nut), 5 seeds (0..4: the 80/20 split of the good training images, and the random changes).
`mvtec_ad_robustness.ipynb`, `helpers.py` and the existing CSV files are not changed. New files: `helpers_followup.py`, `robustness_followup_run.py`,
`robustness_batch_nn_norm.csv` + `robustness_batch_nn_norm_scores/` (Test A), `robustness_batch_nn_refit.csv` + `robustness_batch_nn_refit_scores/` (Test B).

**Reading rule.** The seed spread is the standard deviation over the 5 seeds of the mean over the three objects. A difference counts as real only if it is
clearly larger than that spread. Nothing here says a method is 'robust'; each result says what changed, by how much, and what it cost in detection.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display
pd.set_option("display.width", 200)
A = pd.read_csv("robustness_batch_nn_norm.csv", dtype={"level": str})
OLD = pd.concat([pd.read_csv("robustness_batch.csv", dtype={"level": str}), pd.read_csv("robustness_batch_geometry.csv", dtype={"level": str})]).drop_duplicates(["family", "level", "category"])
TPL = pd.read_csv("robustness_batch_template_photo.csv", dtype={"level": str})
A.groupby("method").size()

## Test A: a normalisation step before feature extraction
Gain and offset fitted per image to the mean of the good training images (`norm_gain`), and the same plus a linear shading term a*x+b*y+c (`norm_shade`). Memory, margin and threshold are fitted as in the baseline, on normalised images. `plain` is the baseline.

### Check: seed 0 of `plain` against the existing results

In [ ]:
K = ["family", "level", "category"]
p = A[(A.method == "plain") & (A.seed == 0)].set_index(K)
o = OLD.set_index(K)
j = p.index.intersection(o.index)
cols = ["auroc", "caught@5%", "frr@5%", "recal_caught@5%", "recal_frr@5%", "caught@1%", "frr@1%"]
print(len(j), "of", len(p), "rows; largest absolute difference:", (p.loc[j, cols] - o.loc[j, cols]).abs().max().max())

### Unchanged images: does normalising cost detection?

In [ ]:
u = A[A.family == "none"]
print("per object, mean and std over the seeds")
display(u.groupby(["category", "method"])[["auroc", "caught@5%", "frr@5%"]].agg(["mean", "std"]).round(3))
print("mean over the objects, then mean and std over the seeds")
display(u.groupby(["method", "seed"])[["auroc", "caught@5%", "frr@5%"]].mean().groupby("method").agg(["mean", "std"]).round(3))

### The 14 changed versions at the strongest level, threshold fixed

In [ ]:
c = A[A.family != "none"]
for col in ["frr@5%", "auroc", "caught@5%"]:
    t = c.groupby(["family", "method", "seed"])[col].mean().unstack("method")
    d = pd.DataFrame({"plain": t.plain.groupby("family").mean(), "gain": t.norm_gain.groupby("family").mean(), "shade": t.norm_shade.groupby("family").mean(),
                      "gain-plain": (t.norm_gain - t.plain).groupby("family").mean(), "std": (t.norm_gain - t.plain).groupby("family").std(),
                      "shade-plain": (t.norm_shade - t.plain).groupby("family").mean(), "std ": (t.norm_shade - t.plain).groupby("family").std()}).round(3)
    print(col)
    display(d)
print("all 14 together, mean and std over the seeds")
display(c.groupby(["method", "seed"])[["auroc", "frr@5%", "caught@5%", "recal_caught@5%", "recal_frr@5%"]].mean().groupby("method").agg(["mean", "std"]).round(3))

### Threshold set again on the changed good images

In [ ]:
for col in ["recal_caught@5%", "recal_frr@5%"]:
    t = c.groupby(["family", "method", "seed"])[col].mean().unstack("method")
    print(col)
    display(pd.DataFrame({"plain": t.plain.groupby("family").mean(), "gain-plain": (t.norm_gain - t.plain).groupby("family").mean(), "std": (t.norm_gain - t.plain).groupby("family").std(),
                          "shade-plain": (t.norm_shade - t.plain).groupby("family").mean(), "std ": (t.norm_shade - t.plain).groupby("family").std()}).round(3))

### Against the normalised template (`robustness_batch_template_photo.csv`, one run, no seeds)

In [ ]:
cc = ["auroc", "frr@5%", "caught@5%", "recal_caught@5%"]
t = TPL[TPL.family != "none"].groupby("family")[cc].mean()
n = c[c.method == "norm_gain"].groupby(["family", "seed"])[cc].mean().groupby("family").mean()
display(t.join(n, lsuffix=" template", rsuffix=" nn gain").round(3))

### Findings, Test A (measured; 5 seeds, 3 objects)

**Unchanged images.** Mean AUROC 0.889 (plain), 0.901 (gain+offset), 0.902 (with shading); spread 0.007, so the rise is borderline. Defects caught at 5 % good parts rejected: 0.663, 0.638, 0.627, with a spread of 0.05-0.10, so the fall is within the spread and is not established as a loss. Good parts rejected on unchanged images: 0.079 -> 0.047, spread 0.03. On transistor alone the caught share falls 0.69 -> 0.62 / 0.59 with a spread of 0.15-0.26: not established, but not excluded.

**Changed images, threshold fixed.** Over the 14 changed versions, good parts rejected 0.261 -> 0.175 (gain) / 0.185 (shading), spread 0.04-0.05; AUROC 0.837 -> 0.857 / 0.850, spread 0.003-0.004 (real, small); defects caught 0.687 -> 0.676 / 0.673 (within the spread). The change in good parts rejected is clearly larger than the spread for contrast_up (0.87 -> 0.21) and brighter (0.46 -> 0.09); about the size of the spread for blur and colour; within it for the other families.
The fall in 'caught' for brighter and contrast_up is not a loss of detection: the unnormalised run caught that much by rejecting almost every part (87 % of good parts for contrast_up).

**Where it is worse.** shift (geometry): good parts rejected 0.56 -> 0.74 (gain) / 0.97 (shading), AUROC 0.715 -> 0.678 / 0.573, both larger than the spread (0.03-0.05). The shading term hurts shift clearly and has no clear advantage elsewhere.

**Threshold set again.** Defects caught 0.558 (plain), 0.565, 0.554: no difference beyond the spread. Normalising therefore mostly removes the need to set the threshold again for lighting changes; it does not raise detection once the threshold is set again. rotate improves a little (+0.09, spread 0.025); shift falls (-0.09 / -0.17).

**Guess (not tested).** The linear shading term may absorb part of the shifted part's edge, which would explain the fall for shift. That has not been checked.

## Test B: refitting the memory with changed good training images

Memories: `orig_only` (the baseline), `refit_mixed` (original + changed copies of darker, brighter, uneven, blur, lens_spots, all three strengths), `refit_only_<family>` (changed copies of that family only). Tested on the strongest level of each of the five families. Margin and threshold from held-out good images changed the same way as the test condition (`recal_*`; margin and threshold both: `reset_*`; the `caught@5%`/`frr@5%` columns keep the threshold of the original held-out images). The memory is capped at 40,000 patches in all cases, so in the refit memories the original patches are a smaller share (this is a confound; a larger cap was not run).

In [ ]:
B = pd.read_csv("robustness_batch_nn_refit.csv", dtype={"level": str})
FAM = ["darker", "brighter", "uneven", "blur", "lens_spots"]
def paired(col):
    rows = []
    for f in FAM:
        base = B[(B.method == "orig_only") & (B.family == f)].groupby("seed")[col].mean()
        r = {"family": f, "orig": base.mean()}
        for m, lab in (("refit_mixed", "mixed"), ("refit_only_" + f, "only")):
            x = B[(B.method == m) & (B.family == f)].groupby("seed")[col].mean()
            d = x - base
            r.update({lab: x.mean(), lab + "-orig": d.mean(), lab + " std": d.std(), lab + " same sign": int((np.sign(d) == np.sign(d.mean())).sum())})
        rows.append(r)
    return pd.DataFrame(rows).set_index("family").round(3)
print(B.groupby("method").seed.nunique().to_dict())

### Good parts rejected and defects caught, threshold of the original held-out images (fixed)

In [ ]:
for col in ["frr@5%", "caught@5%", "auroc"]:
    print(col)
    display(paired(col))

### Margin and threshold set again on the changed held-out good images

In [ ]:
for col in ["recal_caught@5%", "recal_frr@5%"]:
    print(col)
    display(paired(col))

### Unchanged images, and the cost in memory and fitting time

In [ ]:
u = B[B.family == "none"]
base = u[u.method == "orig_only"].groupby("seed")[["auroc", "caught@5%", "frr@5%"]].mean()
rows = {m: ((u[u.method == m].groupby("seed")[["auroc", "caught@5%", "frr@5%"]].mean() - base).agg(["mean", "std"]).stack()) for m in sorted(set(u.method) - {"orig_only"})}
print("unchanged images: refit minus orig_only, mean and std over seeds of the paired difference")
display(pd.DataFrame(rows).T.round(3))
print("memory size and fitting time per object (s)")
display(B.groupby("method")[["n_memory", "fit_s"]].mean().round(0))

### Findings, Test B (measured; 5 seeds, 3 objects, strongest level of each family)

**blur (the clearest effect).** Good parts rejected at the fixed threshold 0.33 -> 0.03 (mixed) / 0.00 (blur-only), paired difference -0.30 / -0.32 with spread 0.12: larger than the spread, same sign in 5 of 5 seeds. AUROC 0.756 -> 0.817 / 0.879 (spread 0.013-0.018). With margin and threshold set again, defects caught 0.438 -> 0.548 (mixed, +0.11, spread 0.04) / 0.656 (blur-only, +0.22, spread 0.04). Cost: defects caught at the fixed threshold fall 0.60 -> 0.36 / 0.19, because the old threshold rejected a third of good parts and now no longer does; with the threshold set again the comparison is the +0.11 / +0.22 above.

**brighter.** Good parts rejected 0.46 -> 0.37 (mixed, -0.09, spread 0.05, 5 of 5 seeds) / 0.19 (brighter-only, -0.27, spread 0.07). AUROC +0.023 with brighter-only (spread 0.013), no change with mixed. Defects caught at the fixed threshold fall (-0.04 mixed, -0.18 only); set again, no difference beyond the spread.

**lens_spots.** Good parts rejected 0.30 -> 0.13 (mixed) / 0.08 (only), -0.18 / -0.23 with spread 0.11-0.13: larger than the spread. AUROC +0.016 (mixed) / +0.042 (only), the second larger than the spread. Set again: defects caught +0.08 (mixed, spread 0.075: about the spread) and +0.02 (only: within the spread).

**darker and uneven.** These did not fail before (good parts rejected 0.03 and 0.09) and nothing is gained: all differences are within the spread, except darker-only (rejected -0.02, spread 0.008; defects caught at the fixed threshold -0.10, spread 0.07) and mixed AUROC -0.02 for both families (spread 0.014-0.017, 5 of 5 seeds), a small loss.

**Cost in detection on unchanged images.** AUROC 0.889 (original memory) -> 0.873 (mixed, -0.016, spread 0.014: about the spread), 0.863 (blur-only, -0.026, spread 0.006: a real small loss), 0.880-0.889 for the other single-family memories. Defects caught at the fixed threshold: mixed unchanged (0.664 against 0.663); single-family memories 0.57-0.62, i.e. 0.04-0.09 lower with spreads of 0.03-0.08 (blur-only about 2 spreads; the others about one).

**Cost in time.** The memory is 40,000 patches in all cases (cap). Fitting time per object 43 s (original), 93-114 s (single family), 442 s (mixed): 2-2.5x and about 10x. Scoring time is the same.

**Not shown by these runs.** Whether the gains hold for changes of a family that was not in the memory (not tested: each memory is tested only on the families it was fitted with, or on the same family). Whether a larger memory cap changes the picture (not run). The single-family memory is tested only on its own family, so the table compares it with the mixed memory on the same footing only for that family.

**Reading.** What changed: a memory that has seen a changed family rejects fewer good parts at the old threshold, clearly for blur, brighter and lens_spots. By how much: 0.2-0.3 fewer good parts rejected for blur and lens_spots. What it cost: a small fall in AUROC on unchanged images (about 0.02, about the spread for mixed) and 2-10x the fitting time. Whether the gain exceeds the spread: yes for blur, brighter-only and lens_spots in good parts rejected; for defects caught with the threshold set again only blur is clearly above the spread. A guess, not tested: the gain for blur is large because a blurred good part is far from every sharp patch in the original memory, so adding blurred copies fills a region of feature space that was empty.